# 04 - Feature Engineering & Model Experiments

This notebook covers **Step 10** (feature engineering) and **Step 11 code** (assembling
the real training dataset) as ▶ FOLLOW NOW sections, and **Step 12 / 14 / 15** as
⏳ DEFER placeholders - present so nothing in the roadmap is skipped over, but
intentionally left as comments only until earlier steps are done and you've checked in.

Requires `notebooks/03_stockfish_analysis.ipynb` to be finished first - this notebook
assumes `moves_clean.parquet` already has evaluation/CPL columns on it.

In [ ]:
import chess
import pandas as pd
import numpy as np
from pathlib import Path

## ▶ Step 10 - Feature Engineering

Build one group at a time. After each group, run `.describe()` and a histogram on the
new columns before moving to the next group - it's much easier to debug one feature
group than five at once.

### 10a. Player features

**What to call:** plain column arithmetic - `df["white_elo"] - df["black_elo"]` for
rating difference, then pick the right sign depending on `player`.

**Why:** rating gap is one of the strongest predictors of decision quality in most
chess-analytics research - cheap to compute, worth getting right first.

In [ ]:
# TODO: rating_difference, player_elo (elo of the player who's actually moving), etc.


### 10b. Time features

**What to call:**
- `df.groupby(["game_id", "player"])["remaining_time"].diff()` for time spent on a move
  (careful with sign, and remember increment adds time back).
- A rolling/expanding mean of that per game+player for "average thinking time".
- A simple threshold (e.g. remaining_time below some cutoff relative to time_control)
  for a time-pressure indicator.

**Why:** this is the feature group the whole project's research question depends on -
spend real time getting it right and sanity-checking it against a few known games.

In [ ]:
# TODO: time_spent_on_move, avg_thinking_time, time_pressure_indicator, time_remaining_ratio


### 10c. Position features

**What to call:** `chess.Board(fen_before)`, then:
- `board.piece_map()` for material balance / piece count
- `len(list(board.legal_moves))` as a rough complexity proxy
- python-chess's king-safety / pawn-structure helpers (check the `chess` module docs -
  linked in `docs/NEXT_STEPS.md` - for what's built in vs. what you compute yourself)

In [ ]:
# TODO: material_balance, piece_count, legal_move_count, king_safety, game_phase


### 10d. Engine features

These should already exist as columns from Step 9/9b (`eval_before`, `eval_after`,
`cpl`). Here you just derive a couple more: evaluation swing (`eval_after - eval_before`)
and whatever "tactical difficulty" proxy you decide on (e.g. legal move count + material
imbalance, or something you research yourself).

In [ ]:
# TODO: eval_swing, tactical_difficulty


### 10e. Behavioural features

**What to call:** `df.groupby(["game_id", "player"])["cpl"].rolling(window=...)` for
rolling average CPL; a shift+compare (`df["cpl"] > threshold` then a rolling sum) for
consecutive errors; `.diff()` again for an evaluation/thinking-time trend.

**Why grouped by game_id AND player:** you want "this player's last few moves in this
game", not moves mixed across different games or leaking the opponent's moves in.

In [ ]:
# TODO: previous_cpl, rolling_avg_cpl, recent_accuracy, consecutive_errors, decision_consistency


## ▶ Step 11 (code) - Assemble the real training dataset

Compare this against the conceptual layout in `docs/training_dataset.md` - this is
where it stops being conceptual.

**What to call:**
- `df[[...]]` with the actual list of feature columns you built above, to form `X`.
- Your Step 9b thresholds, applied to `cpl`, to form `y2` (blunder 0/1).
- A CPL-based scoring formula you design and document, to form `y1` (decision quality
  0-100).

This is a good checkpoint to stop and check in before Step 12.

In [ ]:
# TODO: X = df[[...]]
# TODO: y1 = ...  (decision quality)
# TODO: y2 = ...  (blunder flag)


---
## ⏳ Step 12 - Model Training - DEFER

**Not started.** Per project scope, implementation stops at Step 11 until there's a
clean, reliable feature dataset to train on (which doesn't exist until the sections
above are actually built and validated) - and until your mentor signs off on moving
forward. This section is a placeholder so the notebook maps the roadmap, not an
invitation to start.

When you do get the go-ahead: baseline classifiers (Logistic Regression, Random Forest,
XGBoost) for `y2`, baseline regressors (Linear Regression, Random Forest Regressor,
XGBoost Regressor) for `y1` - compare all of them, don't assume XGBoost wins by
default.

In [ ]:
# DEFERRED - do not implement yet.
# from sklearn.linear_model import LogisticRegression, LinearRegression
# from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
# from xgboost import XGBClassifier, XGBRegressor


## ⏳ Step 14 - Explainable AI (SHAP) - DEFER

**Not started.** SHAP explains an *already-trained* model - nothing to point it at yet.
When Step 12 produces a tree-based model, `shap.TreeExplainer` is the natural starting
point (fast and exact for that model family). Resources in `docs/NEXT_STEPS.md`.

## ⏳ Step 15 - Dashboard - DEFER

**Not started**, and the roadmap says so explicitly: "this is not our immediate
priority. We need the data and model working first." Nothing to build here until
Steps 12 and 14 exist.